<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Final Project · Fit Your Range Model
### Train on what really happened. Forecast with the NWS forecast. A quantile forest gives your range.

*Final project · Colab · your energy project on Google Cloud*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/FinalProject_Fit_Your_Range_Model.ipynb)

**The simple path for the final project.** Your model learns from the Lab 5 data: the weather measured at Bradley
Airport and Connecticut's electricity load, every hour since 2023. Every hour in production, `forecast-24h` feeds it the
**National Weather Service forecast** (your Lab 5 weather scraper), and the model gives three numbers per hour: a
**low**, an **expected** and a **high** load. That's a probabilistic model, and it's what the final project scores.

The model is the **quantile forest** from Lab 6B: a random forest that remembers every training load in its leaves, so
it can give any percentile. Your low is its 10th percentile and your high its 90th, so about 8 hours in 10 should land
between them.

| Part | Time | What you do |
|---|---|---|
| **0. Sync your fork** | 2 min | One click, so `forecast-24h` knows this model |
| **1. Setup** | 3 min | The Lab 5 data, three periods, the leaderboard's score |
| **2. Your first range model** | 5 min | Fit one quantile forest, score it on 2025 |
| **3. Fit and compare** | 10+ min | Try settings, every one scored the same way |
| **4. Calibrate the width** | 3 min | Stretch or shrink the range until it's honest |
| **5. One look at 2026** | 2 min | Before and after, on hours you never tuned on |
| **6. Ship it** | 5 min | Upload, run `forecast-24h`, get your "Final model is live" line |
| **7. Watch it live** | 2 min | Your Inside range on the leaderboard, and how to adjust |

Already shipped a range in Lab 6C or 6D? That model counts too. Lab 6's HRRR weather runs are a correct but more
complex extension of this same work. This notebook is the recommended path, and you don't need HRRR for it.

# Part 0 · Sync your fork (2 min)
Dr. Dave added this model to `forecast-24h` in the class repo. Bring it into **your** fork:

1. Open your fork on GitHub (`github.com/YOUR-USERNAME/energy-forecasting`).
2. Click **Sync fork**, then **Update branch**.
3. Click **Actions**: **Deploy forecast-24h** starts by itself. Wait for the green check (about 3 minutes).

What's new: a `range_method` called **`quantile_forest`**. Your Lab 5 model stays in your bucket as the backup.

# Part 1 · Setup (3 min)
Type your energy project, bucket, NetID and a **name for your model**. The leaderboard shows that name in the
**Live model** column, so you always know which model is being scored. Leave the project as it is for **demo mode**:
everything runs except the upload in Part 6.

In [ ]:
!pip install -q scikit-learn==1.6.1 quantile-forest==1.4.2

import io
import json
import time
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from pandas.tseries.holiday import USFederalHolidayCalendar
from quantile_forest import RandomForestQuantileRegressor

PROJECT_ID = "YOUR-ENERGY-PROJECT-ID"     # e.g. "energy-dww05002-f26"
BUCKET_NAME = "YOUR-ENERGY-PROJECT-ID"    # usually the same as your project ID
NETID = "abc12345"                        # your UConn NetID, lower case
MODEL_NAME = "My quantile forest"         # up to 40 characters: the leaderboard shows it next to your nickname

SEED = 42
LOCAL = "America/New_York"
DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", index_col="hour_utc", parse_dates=["hour_utc"])
print("scikit-learn", sklearn.__version__, "(production uses 1.6.1) |", len(df), "hours of Bradley weather and Connecticut load")

The same tools as Lab 5 and 6B, word for word: `make_features` (the 11 inputs production builds every hour) and the
leaderboard's score, `crps`. Then the three periods. **The rule: tune on 2025, look at 2026 once, at the end.**

In [ ]:
WEATHER = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct"]
CALENDAR = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]
FEATURES = CALENDAR + WEATHER + ["load_yesterday_mw"]


def make_features(hours_utc, weather, load):
    """One row of model inputs per hour (word for word from Lab 5 and forecast-24h)."""
    hours_utc = pd.DatetimeIndex(hours_utc)
    local = hours_utc.tz_convert(LOCAL)
    out = pd.DataFrame(index=hours_utc)
    out["hour"] = local.hour
    out["day_of_week"] = local.dayofweek
    out["month"] = local.month
    out["is_weekend"] = (local.dayofweek >= 5).astype(int)
    holidays = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
    out["is_holiday"] = local.tz_localize(None).normalize().isin(holidays).astype(int)
    for column in WEATHER:
        out[column] = weather[column].reindex(hours_utc).values
    yesterday = load.reindex(hours_utc - pd.Timedelta(hours=24)).values
    hour_before = load.reindex(hours_utc - pd.Timedelta(hours=25)).values
    out["load_yesterday_mw"] = np.where(np.isnan(yesterday), hour_before, yesterday)
    return out[FEATURES]


def crps(actual, low, expected, high):
    """The leaderboard's score for each hour, in MW (lower is better)."""
    actual = np.asarray(actual, dtype=float)
    outside = np.clip(low - actual, 0, None) + np.clip(actual - high, 0, None)
    return (0.5 * np.abs(actual - expected) + 0.1 * (high - low) + 1 * outside) / 1.5


def score_range(low, expected, high, actual):
    """CRPS, the share of hours inside the range, and the average width."""
    actual = np.asarray(actual, dtype=float)
    inside = (actual >= low) & (actual <= high)
    return {"CRPS (MW)": round(float(np.mean(crps(actual, low, expected, high))), 1),
            "inside range (%)": round(100 * float(np.mean(inside)), 1),
            "average width (MW)": round(float(np.mean(high - low))),
            "avg miss (MW)": round(float(np.mean(np.abs(actual - expected))), 1)}


X_all = make_features(df.index, df, df["load_mw"]).iloc[25:]
y_all = df["load_mw"].loc[X_all.index]
years = X_all.index.tz_convert(LOCAL).year
train = years.isin([2023, 2024])
valid = years == 2025
test = years == 2026

periods = pd.DataFrame({"period": ["train", "validation", "test"],
                        "years": ["2023-2024", "2025", "2026 (to Oct 15)"],
                        "hours": [int(train.sum()), int(valid.sum()), int(test.sum())],
                        "used for": ["fitting", "scoring every setting you try", "one look, at the end"]})
periods

# Part 2 · Your first range model (5 min)
One quantile forest, the 6C settings: 100 trees, at least 10 training hours per leaf. For every 2025 hour it gives the
10th, 50th and 90th percentile: your low, expected and high. Scored with the **weather that really happened**, so this
is the model's own uncertainty. Live it only has the NWS forecast, so expect your live score to be a little worse.

In [ ]:
EXPERIMENTS = []          # the log: one row per setting you try
PREDICTIONS = {}          # each setting's 2025 low, expected and high, so Part 4 needs no refit


def try_forest(settings, note):
    """Fit a quantile forest on 2023-2024 with these settings, score it on 2025, and log it."""
    started = time.time()
    forest = RandomForestQuantileRegressor(n_estimators=settings["n_estimators"],
                                           min_samples_leaf=settings["min_samples_leaf"],
                                           max_features=settings["max_features"],
                                           random_state=SEED, n_jobs=-1)
    forest.fit(X_all[train], y_all[train])
    q = forest.predict(X_all[valid], quantiles=[0.1, 0.5, 0.9])
    row = {"note": note}
    row.update(settings)
    row.update(score_range(q[:, 0], q[:, 1], q[:, 2], y_all[valid]))
    row["fit (s)"] = round(time.time() - started)
    EXPERIMENTS.append(row)
    PREDICTIONS[note] = q
    print(note, "| CRPS", row["CRPS (MW)"], "| inside", row["inside range (%)"], "% | width", row["average width (MW)"], "MW")
    return row


def show_log():
    return pd.DataFrame(EXPERIMENTS).sort_values("CRPS (MW)").reset_index(drop=True)


STARTER = {"n_estimators": 100, "min_samples_leaf": 10, "max_features": 1.0}
try_forest(STARTER, "starter (6C settings)")

baseline = df["load_mw"].reindex(X_all.index[valid] - pd.Timedelta(hours=24)).values
print("For comparison, same hour yesterday misses by", round(float(np.nanmean(np.abs(y_all[valid].values - baseline))), 1),
      "MW on average (its CRPS is the same number: no range).")

# Part 3 · Fit and compare (10+ min)
Every setting is scored the same way, on 2025, and written in the log. **Just run the next cell:** it's filled in. It
copies the starter settings, changes one thing, and fits again, for a few leaf sizes and feature shares.

- `min_samples_leaf`: the fewest training hours in a leaf. **Smaller** leaves hold only very similar hours, so the
  range gets **narrower** (maybe too narrow). **Bigger** leaves mix more days: **wider**.
- `max_features`: the share of the 11 inputs each split may look at. Less choice makes the trees more different from
  each other, which often averages better.

In [ ]:
for leaf in [5, 20, 40]:
    trial = dict(STARTER)
    trial["min_samples_leaf"] = leaf
    try_forest(trial, "leaf " + str(leaf))

for share in [0.5, 0.33]:
    trial = dict(STARTER)
    trial["max_features"] = share
    try_forest(trial, "max_features " + str(share))

show_log()

**Your turn.** Copy a block into the next cell, change a number, run it. Each fit takes about 10 to 30 seconds.

Two knobs at once (start from the best leaf and share above):
```python
trial = dict(STARTER)
trial["min_samples_leaf"] = 5
trial["max_features"] = 0.5
try_forest(trial, "leaf 5 + max_features 0.5")
```

More trees (smoother percentiles, bigger file):
```python
trial = dict(STARTER)
trial["n_estimators"] = 200
try_forest(trial, "200 trees")
```

Write down **why** you think a change will help before you run it. A change that loses is a fine result for your
report, as long as you tested it fairly.

**Going further is up to you.** More weather stations (other Connecticut airports), more data, new features: all
welcome, and all yours to build. The data README (`Module4/data/README.md`, "Optional (final project): add more
airports") explains where to get them. The catch: production has to build the same inputs every hour, so a new input
also means changing `make_features` in your fork's `forecast-24h` and its weather scraper.

In [ ]:
# Paste one or more blocks from above here, change the numbers, and run.

show_log()

# Part 4 · Calibrate the width (3 min)
`range_widen` stretches your low and high around the expected value: `1.2` = 20% wider. It lives in the model card,
so changing it later needs **no refit**. **Just run the next cell:** it tries seven widens on your best setting and
picks the lowest 2025 CRPS **among those that keep at least 80% of hours inside**. Why that rule: live, the model only
has the NWS forecast, which is wrong sometimes too, so a range that's just barely honest on 2025 runs a bit narrow live.

In [ ]:
best_note = show_log()["note"].iloc[0]
q = PREDICTIONS[best_note]
rows = []
for widen in [0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    low = q[:, 1] - widen * (q[:, 1] - q[:, 0])
    high = q[:, 1] + widen * (q[:, 2] - q[:, 1])
    row = {"range_widen": widen}
    row.update(score_range(low, q[:, 1], high, y_all[valid]))
    rows.append(row)
widen_table = pd.DataFrame(rows)
# The rule: the lowest CRPS among widens that keep at least 80% of 2025 hours inside (live, ranges run a bit narrow)
honest = widen_table[widen_table["inside range (%)"] >= 80]
if len(honest) == 0:
    honest = widen_table
RANGE_WIDEN = float(honest.sort_values("CRPS (MW)")["range_widen"].iloc[0])
print("Best setting:", best_note, "| range_widen picked (lowest CRPS with 80%+ inside on 2025):", RANGE_WIDEN)
widen_table

# Part 5 · One look at 2026 (2 min)
The starter and your winner, refit on 2023-2025 and scored once on 2026 (to Oct 15), with your widen. 2026 never
picked anything, so this is an honest before and after for your report.

In [ ]:
WINNER = dict(STARTER)
winner_row = show_log().iloc[0]
for name in ["n_estimators", "min_samples_leaf", "max_features"]:
    WINNER[name] = winner_row[name]
WINNER["n_estimators"] = int(WINNER["n_estimators"])
WINNER["min_samples_leaf"] = int(WINNER["min_samples_leaf"])
WINNER["max_features"] = float(WINNER["max_features"])

before_test = train | valid
results = []
for label, settings, widen in [("starter (6C settings)", STARTER, 1.0), ("your winner", WINNER, RANGE_WIDEN)]:
    forest = RandomForestQuantileRegressor(n_estimators=settings["n_estimators"],
                                           min_samples_leaf=settings["min_samples_leaf"],
                                           max_features=settings["max_features"],
                                           random_state=SEED, n_jobs=-1)
    forest.fit(X_all[before_test], y_all[before_test])
    q = forest.predict(X_all[test], quantiles=[0.1, 0.5, 0.9])
    low = q[:, 1] - widen * (q[:, 1] - q[:, 0])
    high = q[:, 1] + widen * (q[:, 2] - q[:, 1])
    row = {"model": label, "range_widen": widen}
    row.update(score_range(low, q[:, 1], high, y_all[test]))
    results.append(row)
TEST_RESULTS = pd.DataFrame(results)
TEST_RESULTS

# Part 6 · Ship it (5 min)
Your winner is refit on **all** the data (2023 to Oct 15, 2026), then uploaded as `energy/models/model_qrf.joblib`
with a model card that says `"range_method": "quantile_forest"`. The cell then runs `forecast-24h` once and checks your
newest forecast. When the **Google sign-in** pop-up opens, pick the Gmail you use for Google Cloud.

In [ ]:
final_forest = RandomForestQuantileRegressor(n_estimators=WINNER["n_estimators"],
                                             min_samples_leaf=WINNER["min_samples_leaf"],
                                             max_features=WINNER["max_features"],
                                             random_state=SEED, n_jobs=-1)
final_forest.fit(X_all, y_all)
joblib.dump(final_forest, "model_qrf.joblib", compress=3)

winner_test = TEST_RESULTS[TEST_RESULTS["model"] == "your winner"].iloc[0]
card = {"model_type": "RandomForestQuantileRegressor",
        "model_name": MODEL_NAME.strip()[:40],
        "notebook": "Final project: fit your range model",
        "range_method": "quantile_forest",
        "range_quantiles": [0.1, 0.9],
        "range_widen": RANGE_WIDEN,
        "weather_in_production": "NWS forecast (scraper-weather-forecast)",
        "trained_on": "Bradley station observations + Connecticut load",
        "best_params": {"n_estimators": WINNER["n_estimators"], "min_samples_leaf": WINNER["min_samples_leaf"],
                        "max_features": WINNER["max_features"]},
        "features": FEATURES,
        "train_period": "2023-01-01 to 2026-10-15 (Connecticut time)",
        "validation_crps_mw_2025": float(show_log()["CRPS (MW)"].iloc[0]),
        "test_crps_mw_2026": float(winner_test["CRPS (MW)"]),
        "test_inside_pct_2026": float(winner_test["inside range (%)"]),
        "test_width_mw_2026": float(winner_test["average width (MW)"]),
        "sklearn_version": sklearn.__version__,
        "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")}
with open("model_card.json", "w") as f:
    json.dump(card, f, indent=1)
print("Saved model_qrf.joblib (" + str(round(len(open("model_qrf.joblib", "rb").read()) / 1e6, 1)) + " MB) and model_card.json")

UPLOAD = "YOUR-ENERGY-PROJECT-ID" not in PROJECT_ID
if not UPLOAD:
    print("DEMO MODE: type your PROJECT_ID and BUCKET_NAME in Part 1 to ship.")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    stamp = card["trained_at"].replace(":", "-")
    for name in ["model_qrf.joblib", "model_card.json"]:
        bucket.blob("energy/models/" + name).upload_from_filename(name)
        bucket.blob("energy/models/history/" + name.replace(".", "_" + stamp + ".", 1)).upload_from_filename(name)
        print("Live: gs://" + BUCKET_NAME + "/energy/models/" + name)

    started = datetime.now(timezone.utc)
    !gcloud scheduler jobs run forecast-24h-hourly --location=us-central1 --project={PROJECT_ID}
    print("Started forecast-24h. Waiting for your forecast", end="")
    latest_blob = bucket.blob("energy/forecasts/latest.csv")
    ready = False
    for attempt in range(16):
        time.sleep(15)
        print(".", end="")
        if latest_blob.exists():
            latest_blob.reload()
            if latest_blob.updated > started:
                ready = True
                break
    print()
    if not ready:
        print("No new forecast after 4 minutes. Look at the log: gcloud functions logs read forecast-24h --region=us-central1 --gen2 --limit=20")
    else:
        latest = pd.read_csv(io.BytesIO(latest_blob.download_as_bytes()))
        method = str(latest["range_method"].iloc[0])
        if method == "quantile_forest":
            width = round(float((latest["predicted_high_mw"] - latest["predicted_low_mw"]).mean()))
            print("Final model is live:", NETID.strip().lower(), "|", card["model_name"], "| leaf", WINNER["min_samples_leaf"],
                  "| max_features", WINNER["max_features"], "| widen", RANGE_WIDEN, "| next 24 h average width", width, "MW")
        else:
            print("range_method came back as", method, "- your forecast-24h doesn't know quantile_forest yet: do Part 0 (Sync fork), wait for the green check, then run this cell again.")

# Part 7 · Watch it live (2 min)
Open the class leaderboard: https://storage.googleapis.com/opim5512-leaderboard-f26/index.html. Within about two
hours your row shows **Final project: quantile forest** and your model name in the **Live model** column. Check your
**Inside range** after a day or two: about 80% is the goal.

- **Well under 80%** (say 65%): your range is too narrow live. Raise `NEW_WIDEN` below and run the cell: it uploads only
  the model card, no refit, and `forecast-24h` uses it at its next run (:20).
- **Well over 90%**: your range is wider than it needs to be. Lower `NEW_WIDEN` a little.

In [ ]:
NEW_WIDEN = 1.3          # change this, then run

if "YOUR-ENERGY-PROJECT-ID" in PROJECT_ID:
    print("DEMO MODE: type your PROJECT_ID in Part 1 first.")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    live_card = json.loads(bucket.blob("energy/models/model_card.json").download_as_text())
    live_card["range_widen"] = NEW_WIDEN
    bucket.blob("energy/models/model_card.json").upload_from_string(json.dumps(live_card, indent=1), content_type="application/json")
    print("range_widen is now", NEW_WIDEN, "- forecast-24h uses it at its next run (:20).")

---

**Data credits:** weather from the Iowa State Mesonet archive of NWS/FAA reports at Bradley; load from ISO New
England. quantile-forest by Zillow (github.com/zillow/quantile-forest).